
# County Rainfall and Planting Advisory

## Project Overview

This project analyses monthly rainfall data from five Kenyan counties: Nairobi, Nakuru, Kisumu, Mombasa and Machakos.

The main objective is to clean rainfall data, calculate annual and monthly rainfall statistics, identify dry months and recommend the best consecutive two-month planting windows.

## Objectives

1. Clean invalid rainfall readings in the Machakos dataset.
2. Combine the five counties into a NumPy array.
3. Calculate annual rainfall totals and monthly averages.
4. Identify the wettest month in each county.
5. Count months receiving less than 50 mm of rainfall.
6. Identify the best consecutive two-month planting window.
7. Generate a CSV summary for the NGO.

## Tools and Libraries

- Python
- NumPy
- CSV module
- Google Colab


In [1]:

import numpy as np
import csv


In [2]:

# County names
counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]

# Month names
months = [
    "January", "February", "March", "April",
    "May", "June", "July", "August",
    "September", "October", "November", "December"
]

# Clean rainfall data for the first four counties
clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],  # Nairobi
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],   # Nakuru
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],  # Kisumu
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70]   # Mombasa
]

# Raw Machakos rainfall data containing errors
machakos_raw = [
    "48", "35", "88", "N/A", "140", " 20 ",
    "9", "12", "", "45", "182", "96"
]

print("Counties:", counties)
print("Number of clean county rows:", len(clean_rows))
print("Machakos raw readings:", machakos_raw)


Counties: ['Nairobi', 'Nakuru', 'Kisumu', 'Mombasa', 'Machakos']
Number of clean county rows: 4
Machakos raw readings: ['48', '35', '88', 'N/A', '140', ' 20 ', '9', '12', '', '45', '182', '96']


In [3]:

def clean_machakos(raw_data, month_names):

    valid_values = []
    invalid_indices = []

    # Convert valid strings into integers
    for index, value in enumerate(raw_data):

        try:
            rainfall = int(value.strip())
            valid_values.append(rainfall)

        except ValueError:
            invalid_indices.append(index)

    # Calculate the rounded average of valid readings
    average_rainfall = round(sum(valid_values) / len(valid_values))

    # Replace invalid readings with the average
    cleaned_data = []

    for index, value in enumerate(raw_data):

        if index in invalid_indices:
            cleaned_data.append(average_rainfall)
        else:
            cleaned_data.append(int(value.strip()))

    # Display the imputed months
    imputed_months = [month_names[i] for i in invalid_indices]

    print("Valid rainfall average:", average_rainfall, "mm")
    print("Imputed months:", imputed_months)
    print("Cleaned Machakos data:", cleaned_data)

    return cleaned_data


# Clean Machakos rainfall readings
machakos_clean = clean_machakos(machakos_raw, months)


Valid rainfall average: 68 mm
Imputed months: ['April', 'September']
Cleaned Machakos data: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]


In [4]:

# Combine the five counties
all_rows = clean_rows + [machakos_clean]

# Convert the combined data into a NumPy array
R = np.array(all_rows)

# Display the rainfall array
print("Rainfall Data:")
print(R)

# Display the shape and data type
print("\nArray shape:", R.shape)
print("Array dtype:", R.dtype)


Rainfall Data:
[[ 64  56  92 219 176  35  18  24  31  61 165 104]
 [ 35  42  70 145 120  78  96 105  72  55  60  38]
 [ 58  90 150 240 180  95  70  85  90  88 130  95]
 [ 30  15  55 150 240 110  90  65  60  90 100  70]
 [ 48  35  88  68 140  20   9  12  68  45 182  96]]

Array shape: (5, 12)
Array dtype: int64


In [5]:

# Calculate annual rainfall totals for each county
annual_totals = np.sum(R, axis=1)

print("Annual Rainfall Totals")

for county, total in zip(counties, annual_totals):
    print(f"{county}: {total} mm")


Annual Rainfall Totals
Nairobi: 1045 mm
Nakuru: 916 mm
Kisumu: 1371 mm
Mombasa: 1075 mm
Machakos: 811 mm


In [6]:

# Calculate average monthly rainfall across all counties
monthly_averages = np.mean(R, axis=0)

print("Average Monthly Rainfall")

for month, average in zip(months, monthly_averages):
    print(f"{month}: {average:.1f} mm")


Average Monthly Rainfall
January: 47.0 mm
February: 47.6 mm
March: 91.0 mm
April: 164.4 mm
May: 171.2 mm
June: 67.6 mm
July: 56.6 mm
August: 58.2 mm
September: 64.2 mm
October: 67.8 mm
November: 127.4 mm
December: 80.6 mm


In [7]:

# Find the index of the wettest month for each county
wettest_indices = np.argmax(R, axis=1)

# Convert the indices into month names
wettest_months = [months[i] for i in wettest_indices]

# Pair counties with their wettest months using zip()
print("Wettest Month per County")

for county, wettest_month in zip(counties, wettest_months):
    print(f"{county}: {wettest_month}")


Wettest Month per County
Nairobi: April
Nakuru: April
Kisumu: April
Mombasa: May
Machakos: November


In [8]:

# Create a Boolean mask for rainfall below 50 mm
dry_mask = R < 50

# Count the dry months for each county
dry_month_counts = np.sum(dry_mask, axis=1)

print("Number of Dry Months per County")

for county, count in zip(counties, dry_month_counts):
    print(f"{county}: {count} dry months")


Number of Dry Months per County
Nairobi: 4 dry months
Nakuru: 3 dry months
Kisumu: 0 dry months
Mombasa: 2 dry months
Machakos: 6 dry months


In [9]:

# Calculate combined rainfall for consecutive months
two_month_rainfall = R[:, :-1] + R[:, 1:]

# Find the index of the highest two-month rainfall
planting_indices = np.argmax(two_month_rainfall, axis=1)

# Convert the indices into month ranges
planting_windows = [
    f"{months[i]}-{months[i + 1]}"
    for i in planting_indices
]

# Display the recommended planting windows
print("Recommended Planting Windows")

for county, window in zip(counties, planting_windows):
    print(f"{county}: {window}")


Recommended Planting Windows
Nairobi: April-May
Nakuru: April-May
Kisumu: April-May
Mombasa: April-May
Machakos: November-December


In [10]:

# Create the CSV summary file
with open("rainfall_summary.csv", "w", newline="") as file:

    writer = csv.writer(file)

    # Write the column headings
    writer.writerow([
        "county",
        "annual_total_mm",
        "wettest_month",
        "dry_months",
        "planting_window"
    ])

    # Write one row for each county
    for row in zip(
        counties,
        annual_totals,
        wettest_months,
        dry_month_counts,
        planting_windows
    ):
        writer.writerow(row)

print("rainfall_summary.csv created successfully!")


rainfall_summary.csv created successfully!


In [11]:

# Read and display the generated CSV
with open("rainfall_summary.csv", "r") as file:
    print(file.read())


county,annual_total_mm,wettest_month,dry_months,planting_window
Nairobi,1045,April,4,April-May
Nakuru,916,April,3,April-May
Kisumu,1371,April,0,April-May
Mombasa,1075,May,2,April-May
Machakos,811,November,6,November-December




## Planting Advisory

Based on the rainfall analysis, Nairobi, Nakuru, Kisumu and Mombasa have their highest consecutive two-month rainfall totals during April and May, making this the recommended planting window for these counties. Machakos has its highest combined rainfall during November and December, making this the recommended planting window for farmers in that county. Kisumu recorded the highest annual rainfall at 1,371 mm, while Machakos recorded the lowest at 811 mm. Farmers in Machakos should take extra precautions because the county recorded six dry months, compared to zero in Kisumu. The NGO should use these findings to guide county-specific SMS planting messages while encouraging farmers to consider local soil conditions, crop varieties and access to water. These recommendations are based on one year's monthly rainfall readings and should not be treated as guaranteed forecasts of future rainfall.



## Data Quality Notes

The original Machakos rainfall dataset contained two invalid entries: "N/A" in April and an empty string in September. These values were identified using exception handling with a specific `ValueError` exception. The average of the ten valid Machakos readings was 67.5 mm, which was rounded to 68 mm and used to replace both invalid readings. The imputation ensures that the dataset contains five complete county records with twelve monthly rainfall values each. However, replacing missing readings with the average may not represent the actual rainfall received during those months, and this should be considered when interpreting the planting recommendations.
